# Train and backtest equities and synthetic options: $1T, $100B, $10B

This notebook explains and runs the current warehouse-streaming workflow. Change the universe in **Step 1**; the same trainer, model, document rules, and backtests serve all three thresholds.

**Flow:** configuration → fresh warehouse discovery → annual equity/option documents on demand → GPU optimization → predictions without warmup → four separate backtest books → coverage and results.

The saved notebook opens in **review** mode, so Run All displays existing artifacts without starting another training job. To train, set `MODE = "train"` and choose one or more `UNIVERSES`. Runs execute sequentially. Review reads saved outputs only; fresh training never consumes them as a corpus.

Requirements: a kernel with this repository's dependencies, access to the populated `quant-warehouse` store, and CUDA for training. Follow the repository environment instructions; `quant-warehouse` must come from `quantarb/quant-warehouse` on `main`. This notebook does not download data or install packages.

## Kernel dependency setup

This notebook needs the current `quant-warehouse` package layout, including
`quant_warehouse.platforms`. An older build can have the same version number
but lack that module. The cell below prints the active Python and package paths.
If that API is missing, it reinstalls the package from the repository's `main`
branch using **this kernel's Python**, preserving other installed dependencies.

After a repair, **restart the kernel and Run All** so previously imported modules
are cleared. An already compatible environment is left unchanged. If the install
fails, its pip error is shown directly (for example, missing GitHub access).

The check tests the full import in a fresh Python process as well as the running
kernel. If the installed files work but the kernel retains old modules, it asks
for **Kernel → Restart Kernel and Run All Cells**, without reinstalling again.
Rerunning a cell or refreshing the browser does not restart Python.

In [1]:
import sys
import subprocess
import importlib.util

print("Kernel Python:", sys.executable)
warehouse_spec = importlib.util.find_spec("quant_warehouse")
print("quant_warehouse:", warehouse_spec.origin if warehouse_spec else "not installed")
api_probe = (
    "from quant_warehouse.warehouse.backend import FrameFormat, StorageBackend; "
    "from quant_warehouse.platforms.data_providers.thetadata.options "
    "import read_thetadata_eod_option_chain"
)
# A separate process checks files on disk without the kernel's cached modules.
probe = subprocess.run([sys.executable, "-c", api_probe], capture_output=True, text=True)
if probe.returncode:
    print("Repairing quant-warehouse in this kernel's environment...", flush=True)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
        "quant-warehouse @ git+https://github.com/quantarb/quant-warehouse.git@main",
    ])
    probe = subprocess.run([sys.executable, "-c", api_probe], capture_output=True, text=True)
    if probe.returncode:
        raise RuntimeError("Warehouse import still fails in a fresh process:\n" + probe.stderr)
try:
    from quant_warehouse.warehouse.backend import FrameFormat, StorageBackend
    from quant_warehouse.platforms.data_providers.thetadata.options import read_thetadata_eod_option_chain
except ImportError:
    raise RuntimeError(
        "The installed package passes in a fresh process, but this kernel still has old "
        "quant_warehouse modules in memory. In Jupyter choose Kernel > Restart Kernel "
        "and Run All Cells. Rerunning this cell or refreshing the browser does not "
        "restart Python. No further package installation is needed."
    ) from None
print("Warehouse option-chain API is available.")

Kernel Python: /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python
quant_warehouse: /home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/quant_warehouse/__init__.py


Warehouse option-chain API is available.


## Step 1 — Choose the universe and dates

`min_market_cap` is an underlying-symbol filter, in USD. The loader queries stored US NYSE/NASDAQ profiles with ETF/fund exclusion flags, then requires at least two pre-cutoff equity prices. It discovers stored option dates independently for each selected share class. The resulting roster depends on warehouse metadata; it is **not** a reconstructed historical market-cap universe for each year.

Training uses calendar years before January 1, 2024. Scoring covers 2024 through September 9, 2026. Keep these dates fixed when comparing universes. Select `["1T"]`, `["100B"]`, `["10B"]`, or several thresholds. Each training launch gets a new output directory.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import uuid
import pandas as pd
from IPython.display import display

MODE = "review"                         # "train" launches fresh training + backtests
UNIVERSES = ["1T", "100B", "10B"]       # use ["1T"] for the smallest run
MARKET_CAPS = {"1T": 1_000_000_000_000, "100B": 100_000_000_000, "10B": 10_000_000_000}
EPOCHS = 1
BATCH_SIZE = 16
TRAIN_END = "2024-01-01"                 # exclusive training cutoff; must be January 1
PREDICTION_START = "2024-01-01"
PREDICTION_END = "2026-09-09"
# Optional exact output paths for review; otherwise select the latest warehouse-stream run.
REVIEW_RUNS = {}

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_multirate_mtl.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the quant-orchestrator checkout.")
# Import this checkout even when Jupyter starts in notebooks/.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
EXPERIMENT = ROOT / "artifacts/multirate_recovery"
if MODE not in {"review", "train"} or not UNIVERSES or len(set(UNIVERSES)) != len(UNIVERSES):
    raise ValueError("Choose train/review and a nonempty list of distinct universes.")
if set(UNIVERSES) - MARKET_CAPS.keys():
    raise ValueError("Supported universes: 1T, 100B, 10B")
if EPOCHS < 1 or BATCH_SIZE < 1:
    raise ValueError("Epochs and batch size must be positive.")
cutoff, start, end = map(datetime.fromisoformat, [TRAIN_END, PREDICTION_START, PREDICTION_END])
if (cutoff.month, cutoff.day) != (1, 1) or not cutoff <= start <= end:
    raise ValueError("Use a January 1 cutoff <= prediction start <= prediction end.")
display(pd.DataFrame([{"universe": u, "min_market_cap_USD": MARKET_CAPS[u],
                       "mode": MODE, "epochs": EPOCHS} for u in UNIVERSES]))

,universe,min_market_cap_USD,mode,epochs
0,1T,1000000000000,review,1
1,100B,100000000000,review,1
2,10B,10000000000,review,1


## Option EDA — Contracts per symbol on the first trading day

Choose a year and market-cap threshold below. This cell queries the warehouse's
current profile universe and reads each symbol's **full stored option chain** on
that year's first NYSE trading session. It counts distinct contract identifiers
across **all strikes, expirations, calls, and puts**, before synthetic-basket
selection or quote-quality filtering. It does not build a training corpus.

The summary reports the average contracts per symbol **among symbols with a
stored chain**, plus the number of symbols missing that day's chain. Missing
chains are shown as missing, not treated as evidence of zero listed contracts.
The per-symbol table includes call and put counts. This EDA uses the profile
filter directly, without the trainer's additional pre-cutoff equity-price gate.
Change `EDA_YEAR` to compare another year; no later-date fallback is used.

The cell first previews five stored ThetaData option rows from the first symbol
with data, showing **every column without column truncation**. Change
`PREVIEW_ROWS` for a larger sample. These are warehouse-stored ThetaData records,
including normalized and derived fields, before the EDA expiration filter.

Each filtering cell displays the **same summary and per-symbol table**, in the
same original-count order. `before` is the preceding stage, `remaining` is this
stage, and `retained_pct_original` always uses the unfiltered first-day count.
`removed_this_step` includes unknown outcomes excluded at that stage; `unknown`
shows that subset separately. Missing chains remain missing, while a valid chain
with no surviving contracts shows zero. Run the cells in order after edits.

In [3]:
import exchange_calendars as xcals
import polars as pl
from quant_warehouse import Warehouse
from quant_warehouse.platforms.data_providers.thetadata.options import read_thetadata_eod_option_chain

EDA_UNIVERSE = UNIVERSES[1]  # "1T", "100B", or "10B"
EDA_YEAR = 2024
PREVIEW_ROWS = 5

warehouse = Warehouse()
calendar = xcals.get_calendar("XNYS", start=f"{EDA_YEAR - 1}-12-01", end=f"{EDA_YEAR}-12-31")
first_session = calendar.sessions_in_range(f"{EDA_YEAR}-01-01", f"{EDA_YEAR}-01-10")[0]
first_day = first_session.to_pydatetime().replace(tzinfo=None)
profiles = warehouse.catalog.query_symbol_profiles(
    provider="fmp", min_market_cap=MARKET_CAPS[EDA_UNIVERSE], country="US",
    exchanges=["NASDAQ", "NYSE"], exclude_etf=True, exclude_fund=True)
symbols = sorted({profile.symbol for profile in profiles})
first_day_frames = []
contract_rows = []
option_data_preview = None
for symbol in symbols:
    chain = read_thetadata_eod_option_chain(
        symbol, start_date=first_day, end_date=first_day, backend=warehouse.backend)
    if chain.is_empty():
        contract_rows.append({"symbol": symbol, "contracts": None, "calls": None,
                              "puts": None, "status": "missing stored chain"})
        continue
    if option_data_preview is None:
        option_data_preview = chain.sort("contract_symbol").head(PREVIEW_ROWS).to_pandas()
        print(f"ThetaData stored option data: {symbol}, {first_day.date()}, "
              f"{chain.height:,} rows, {chain.width} columns")
        print("All columns:", ", ".join(chain.columns))
        with pd.option_context("display.max_columns", None, "display.max_colwidth", None,
                               "display.width", None, "display.max_rows", None):
            display(option_data_preview)
    contracts = chain.filter(pl.col("contract_symbol").is_not_null()).unique("contract_symbol")
    first_day_frames.append(contracts)
    contract_rows.append({"symbol": symbol, "contracts": contracts.height,
        "calls": contracts.filter(pl.col("option_type") == "call").height,
        "puts": contracts.filter(pl.col("option_type") == "put").height,
        "status": "stored chain"})

if option_data_preview is None:
    print("No stored option data is available for this universe and date.")

# These rows are the shared input to the stacked filters below.
all_options = (pl.concat(first_day_frames, how="diagonal_relaxed") if first_day_frames else
               pl.DataFrame(schema={"underlying_symbol": pl.String, "contract_symbol": pl.String,
                                    "expiration": pl.Datetime("ns"), "strike": pl.Float64, "option_type": pl.String}))
filtered_options = all_options

option_contract_counts = pd.DataFrame(contract_rows, columns=["symbol", "contracts", "calls", "puts", "status"])
for column in ["contracts", "calls", "puts"]:
    option_contract_counts[column] = option_contract_counts[column].astype("Int64")
observed = option_contract_counts.loc[option_contract_counts["status"] == "stored chain", "contracts"]
option_contract_summary = pd.DataFrame([{
    "universe": EDA_UNIVERSE, "first_trading_day": first_day.date().isoformat(),
    "selected_symbols": len(symbols), "symbols_with_chain": len(observed),
    "symbols_missing_chain": len(symbols) - len(observed),
    "average_contracts_per_observed_symbol": observed.mean(),
    "median_contracts_per_observed_symbol": observed.median() if len(observed) else None,
    "min_contracts": observed.min(), "max_contracts": observed.max(),
}])

def show_option_filter(stage, current, previous):
    # Keep the original symbol order and schema at every stage.
    baseline = option_contract_counts.sort_values(
        ["contracts", "symbol"], ascending=[False, True], na_position="last")
    table = baseline[["symbol", "status", "contracts"]].rename(columns={"contracts": "original"})
    table = table.merge(previous[["symbol", "remaining"]].rename(columns={"remaining": "before"}),
                        on="symbol", how="left", validate="one_to_one")
    table = table.merge(current[["symbol", "remaining", "calls", "puts", "unknown"]],
                        on="symbol", how="left", validate="one_to_one")
    observed = table["status"] == "stored chain"
    for column in ["original", "before", "remaining", "calls", "puts", "unknown"]:
        table.loc[observed, column] = table.loc[observed, column].fillna(0)
        table.loc[~observed, column] = pd.NA
        table[column] = table[column].astype("Int64")
    table["removed_this_step"] = table["before"] - table["remaining"]
    table["removed_total"] = table["original"] - table["remaining"]
    table["retained_pct_original"] = 100 * table["remaining"] / table["original"].replace(0, pd.NA)
    table = table[["symbol", "status", "original", "before", "remaining", "removed_this_step",
                   "removed_total", "retained_pct_original", "calls", "puts", "unknown"]]
    summary = {"stage": stage, "symbols_with_chain": int(observed.sum()),
               "symbols_missing_chain": int((~observed).sum())}
    for column in ["original", "before", "remaining", "removed_this_step", "removed_total", "calls", "puts", "unknown"]:
        summary[column] = table[column].sum(min_count=1)
    summary["average_remaining"] = table.loc[observed, "remaining"].mean()
    summary["retained_pct_original"] = (100 * summary["remaining"] / summary["original"]
                                        if pd.notna(summary["original"]) and summary["original"] > 0 else pd.NA)
    print(stage)
    formats = {"retained_pct_original": "{:.2f}%", "average_remaining": "{:,.2f}"}
    display(pd.DataFrame([summary]).style.format(formats, na_rep="missing"))
    display(table.style.format({"retained_pct_original": "{:.2f}%"}, na_rep="missing"))
    return table

baseline_filter_counts = option_contract_counts.rename(columns={"contracts": "remaining"}).assign(unknown=0)
all_options_comparison = show_option_filter("All first-day options", baseline_filter_counts, baseline_filter_counts)


ThetaData stored option data: AAPL, 2024-01-02, 3,792 rows, 43 columns
All columns: date, quote_timestamp, underlying_symbol, expiration, strike, option_type, created_at, last_trade_time, open_price, high_price, low_price, last_trade_price, volume, count, bid_size, bid_exchange, bid, bid_condition, ask_size, ask_exchange, ask, ask_condition, snapshot_date, contract_symbol, data_interval, underlying_price, eod_date, dte, contract_size, theoretical_price, mark, close, close_size, change, change_percent, iv, delta, gamma, theta, vega, rho, mid, open_interest


,date,quote_timestamp,underlying_symbol,expiration,strike,option_type,created_at,last_trade_time,open_price,high_price,low_price,last_trade_price,volume,count,bid_size,bid_exchange,bid,bid_condition,ask_size,ask_exchange,ask,ask_condition,snapshot_date,contract_symbol,data_interval,underlying_price,eod_date,dte,contract_size,theoretical_price,mark,close,close_size,change,change_percent,iv,delta,gamma,theta,vega,rho,mid,open_interest
0,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,50.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,134.75,NaN,15.0,47,135.9,NaN,2024-01-02,AAPL240105C00050000,eod,185.64,2024-01-02,3.0,100.0,135.325,135.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,135.325,NaN
1,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,60.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,124.75,NaN,3.0,60,125.9,NaN,2024-01-02,AAPL240105C00060000,eod,185.64,2024-01-02,3.0,100.0,125.325,125.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,125.325,NaN
2,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,65.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,7,119.75,NaN,5.0,60,120.9,NaN,2024-01-02,AAPL240105C00065000,eod,185.64,2024-01-02,3.0,100.0,120.325,120.325,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,120.325,NaN
3,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,70.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,60.0,1,115.15,NaN,30.0,11,115.9,NaN,2024-01-02,AAPL240105C00070000,eod,185.64,2024-01-02,3.0,100.0,115.525,115.525,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,115.525,NaN
4,2024-01-02 00:00:00.004907,2024-01-02,AAPL,2024-01-05,75.0,call,2262-04-11,2262-04-11,0.0,0.0,0.0,0.0,0.0,NaN,30.0,11,109.90,NaN,12.0,11,110.9,NaN,2024-01-02,AAPL240105C00075000,eod,185.64,2024-01-02,3.0,100.0,110.400,110.400,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,110.400,NaN


All first-day options


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,All first-day options,113,19,166949,166949,166949,0,0,83497,83452,0,"1,477.42",100.00%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,7704,7704,0,0,100.00%,3852,3852,0
1,BKNG,stored chain,6180,6180,6180,0,0,100.00%,3090,3090,0
2,NVDA,stored chain,5492,5492,5492,0,0,100.00%,2746,2746,0
3,META,stored chain,4098,4098,4098,0,0,100.00%,2049,2049,0
4,TSLA,stored chain,3910,3910,3910,0,0,100.00%,1955,1955,0
5,AAPL,stored chain,3792,3792,3792,0,0,100.00%,1896,1896,0
6,NFLX,stored chain,3772,3772,3772,0,0,100.00%,1886,1886,0
7,AMZN,stored chain,3532,3532,3532,0,0,100.00%,1766,1766,0
8,AMD,stored chain,3512,3512,3512,0,0,100.00%,1756,1756,0
9,BA,stored chain,3468,3468,3468,0,0,100.00%,1734,1734,0


## Option filtering — Keep expirations in the selected year

Keep contracts where **`expiration.year == EDA_YEAR`**. For the first trading
session of 2024, this keeps all 2024 expirations, regardless of when the contract
first appeared or traded. There is no origination-date condition.

The expiration-year table shows where the full first-day chain falls before
filtering. The per-symbol table compares counts before and after filtering.
Unknown expiration dates are excluded and counted separately. Averages include
symbols with stored chains, even if their retained count is zero; missing chains
remain missing. `option_expiration_audit` contains each candidate and its keep flag.

This EDA filter does not change the trainer or warehouse storage.

**Stacked input/output:** `all_options` → `same_year_options`. This cell filters
the first cell's saved rows rather than reading a new chain. `filtered_options`
is updated to this stage's survivors.

In [4]:
# Stage 1 consumes the saved first-day rows; it does not reload the universe.
expiration_input = all_options
same_year_options = expiration_input.filter(pl.col("expiration").dt.year() == EDA_YEAR)
filtered_options = same_year_options
expiration_audits = []
filtered_rows = []
for symbol in symbols:
    chain = expiration_input.filter(pl.col("underlying_symbol") == symbol)
    if chain.is_empty():
        filtered_rows.append({"symbol": symbol, "status": "missing stored chain"})
        continue
    contracts = chain.filter(pl.col("contract_symbol").is_not_null()).unique("contract_symbol")
    annotated = contracts.with_columns(
        (pl.col("expiration").dt.year() == EDA_YEAR).fill_null(False).alias("keep_same_year"))
    expiration_audits.append(annotated.select(
        "underlying_symbol", "contract_symbol", "expiration", "strike", "option_type", "keep_same_year"))
    kept = annotated.filter(pl.col("keep_same_year"))
    unknown = annotated["expiration"].null_count()
    filtered_rows.append({
        "symbol": symbol, "status": "stored chain", "before": contracts.height,
        "kept_same_year": kept.height, "removed": contracts.height - kept.height,
        "removed_different_year": contracts.height - kept.height - unknown,
        "removed_unknown_expiration": unknown,
        "calls_kept": kept.filter(pl.col("option_type") == "call").height,
        "puts_kept": kept.filter(pl.col("option_type") == "put").height,
    })

count_columns = ["before", "kept_same_year", "removed", "removed_different_year",
                 "removed_unknown_expiration", "calls_kept", "puts_kept"]
same_year_contract_counts = pd.DataFrame(filtered_rows, columns=["symbol", "status", *count_columns])
for column in count_columns:
    same_year_contract_counts[column] = same_year_contract_counts[column].astype("Int64")
observed_same_year = same_year_contract_counts.loc[same_year_contract_counts["status"] == "stored chain"]
same_year_summary = pd.DataFrame([{
    "universe": EDA_UNIVERSE, "chain_day": first_day.date().isoformat(),
    "symbols_with_chain": len(observed_same_year),
    "symbols_missing_chain": len(symbols) - len(observed_same_year),
    "average_before": observed_same_year["before"].mean(),
    "average_after": observed_same_year["kept_same_year"].mean(),
    "total_kept": observed_same_year["kept_same_year"].sum(min_count=1),
    "total_removed": observed_same_year["removed"].sum(min_count=1),
}])
expiration_filter_counts = same_year_contract_counts.rename(columns={
    "kept_same_year": "remaining", "calls_kept": "calls", "puts_kept": "puts",
    "removed_unknown_expiration": "unknown"})
expiration_comparison = show_option_filter(
    f"Expiration year = {EDA_YEAR}", expiration_filter_counts, all_options_comparison)

option_expiration_audit = pl.concat(expiration_audits) if expiration_audits else pl.DataFrame()
if not option_expiration_audit.is_empty():
    expiration_year_counts = (option_expiration_audit
        .with_columns(pl.col("expiration").dt.year().alias("expiration_year"))
        .group_by("expiration_year").len(name="contracts").sort("expiration_year")
        .with_columns((100 * pl.col("contracts") / pl.col("contracts").sum()).alias("percent")))
    display(expiration_year_counts.to_pandas())


Expiration year = 2024


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,Expiration year = 2024,113,19,166949,166949,136703,30246,30246,68374,68329,0,"1,209.76",81.88%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,7704,5940,1764,1764,77.10%,2970,2970,0
1,BKNG,stored chain,6180,6180,5348,832,832,86.54%,2674,2674,0
2,NVDA,stored chain,5492,5492,4056,1436,1436,73.85%,2028,2028,0
3,META,stored chain,4098,4098,3006,1092,1092,73.35%,1503,1503,0
4,TSLA,stored chain,3910,3910,3114,796,796,79.64%,1557,1557,0
5,AAPL,stored chain,3792,3792,2796,996,996,73.73%,1398,1398,0
6,NFLX,stored chain,3772,3772,2974,798,798,78.84%,1487,1487,0
7,AMZN,stored chain,3532,3532,2808,724,724,79.50%,1404,1404,0
8,AMD,stored chain,3512,3512,2856,656,656,81.32%,1428,1428,0
9,BA,stored chain,3468,3468,2776,692,692,80.05%,1388,1388,0


,expiration_year,contracts,percent
0,2024,136703,81.883090
1,2025,20984,12.569108
2,2026,9262,5.547802


## Option filtering — Finish in the money at expiration

From the same-year-expiration candidates, keep **calls with underlying spot above
strike** and **puts with underlying spot below strike** on the expiration trading
day. At-the-money contracts are excluded. This is an **in-the-money** filter,
not a net-profit test: an option can pay intrinsic value and still lose money
relative to its purchase premium.

Use the last NYSE session on or before expiration, including Friday for Saturday
expirations. The underlying mark is the median finite positive `underlying_price`
in that underlying's stored ThetaData chain **on that exact session**. This is
an EOD proxy, not an official contract-specific settlement fixing; no earlier
quote is carried forward. Forward stock splits between entry and expiration
adjust the strike basis. Reverse splits or invalid split ratios are marked
unknown rather than guessing an adjusted deliverable.

Missing expiration-day prices and unsupported terms are counted as **unknown**,
not as out-of-the-money. Only confirmed in-the-money contracts survive. This uses
future outcomes for retrospective EDA and does not alter training or become an
entry-time trading signal. `option_itm_audit` exposes each decision and its inputs.

The percentage table uses all same-year candidates **before** the ITM filter as
its denominator. ITM, OTM, exactly ATM, and unknown are separate categories;
their percentages sum to 100%. Unknown outcomes are never counted as OTM.

**Stacked input/output:** `same_year_options` → `itm_options`. Expiration marks
are loaded only to evaluate those candidates; no removed contract can re-enter.
`filtered_options` contains the final surviving first-day rows with their original
columns, ready for the next filter. Named stage outputs make rerunning a cell
repeatable. After changing an earlier stage, rerun its downstream cells.

In [5]:
from quant_warehouse.platforms.data_providers.thetadata.options import read_option_chain_arctic

# Stage 2 consumes only the survivors of Stage 1.
itm_input = same_year_options
itm_audits = []
if not itm_input.is_empty():
    candidates = itm_input
    for (symbol,), contracts in candidates.group_by("underlying_symbol", maintain_order=True):
        expirations = contracts["expiration"].unique().to_list()
        sessions = {expiry: calendar.date_to_session(
            expiry.date().isoformat(), direction="previous").to_pydatetime().replace(tzinfo=None)
            for expiry in expirations}
        history = read_option_chain_arctic(
            symbol, start_date=min(sessions.values()), end_date=max(sessions.values()),
            columns=["snapshot_date", "underlying_price"], backend=warehouse.backend)
        spots = {}
        if not history.is_empty() and "underlying_price" in history.columns:
            valid_spots = (history.filter(pl.col("underlying_price").is_finite() &
                                         (pl.col("underlying_price") > 0))
                           .group_by("snapshot_date").agg(pl.col("underlying_price").median().alias("spot")))
            spots = dict(valid_spots.select("snapshot_date", "spot").iter_rows())
        del history
        splits = warehouse.read_fundamentals(
            symbol, section="historical_splits", start=first_day.date().isoformat(),
            end=max(sessions.values()).date().isoformat())
        split_rows = splits.to_dicts() if not splits.is_empty() else []
        settlement_rows = []
        for expiry, session in sessions.items():
            factor, supported = 1.0, True
            for split in split_rows:
                if first_day < split["date"] <= session:
                    numerator, denominator = split.get("numerator"), split.get("denominator")
                    if numerator is None or denominator is None or not (numerator > 0 and denominator > 0):
                        supported = False
                        continue
                    ratio = numerator / denominator
                    if not (1 <= ratio < float("inf")):
                        supported = False
                    else:
                        factor *= ratio
            settlement_rows.append({"expiration": expiry, "expiration_session": session,
                "expiration_spot": spots.get(session), "split_factor": factor,
                "supported_split": supported})
        settlements = pl.DataFrame(settlement_rows).with_columns(
            pl.col("expiration").cast(contracts.schema["expiration"]),
            pl.col("expiration_spot").cast(pl.Float64))
        audit = contracts.join(settlements, on="expiration", how="left").with_columns(
            (pl.col("strike") / pl.col("split_factor")).alias("expiration_strike"))
        known = (pl.col("expiration_spot").is_not_null() & pl.col("supported_split") &
                 pl.col("strike").is_finite() & pl.col("option_type").is_in(["call", "put"]))
        intrinsic = (pl.when(pl.col("option_type") == "call")
                     .then(pl.col("expiration_spot") - pl.col("expiration_strike"))
                     .otherwise(pl.col("expiration_strike") - pl.col("expiration_spot"))).clip(lower_bound=0)
        audit = audit.with_columns(
            pl.when(known).then(intrinsic).otherwise(None).alias("intrinsic_per_adjusted_share"))
        audit = audit.with_columns(
            pl.when(~pl.col("supported_split")).then(pl.lit("unknown_split_adjustment"))
            .when(pl.col("intrinsic_per_adjusted_share").is_null()).then(pl.lit("unknown_expiration_data"))
            .when(pl.col("intrinsic_per_adjusted_share") > 0).then(pl.lit("in_the_money"))
            .otherwise(pl.lit("out_or_at_the_money")).alias("expiry_status"))
        itm_audits.append(audit)

option_itm_audit = pl.concat(itm_audits, how="diagonal_relaxed") if itm_audits else pl.DataFrame()
if option_itm_audit.is_empty():
    print("No same-year candidates to evaluate.")
    empty_itm_counts = pd.DataFrame(columns=["symbol", "remaining", "calls", "puts", "unknown"])
    itm_comparison = show_option_filter("In the money at expiration", empty_itm_counts, expiration_comparison)
else:
    per_symbol_itm = option_itm_audit.group_by("underlying_symbol").agg(
        pl.len().alias("before_itm_filter"),
        (pl.col("expiry_status") == "in_the_money").sum().alias("kept_itm"),
        (pl.col("expiry_status") == "out_or_at_the_money").sum().alias("removed_otm_or_atm"),
        pl.col("expiry_status").str.starts_with("unknown").sum().alias("unknown"))
    # Include symbols with an observed chain but no same-year candidates as zeros.
    base_symbols = same_year_contract_counts.loc[
        same_year_contract_counts["status"] == "stored chain", ["symbol"]].rename(columns={"symbol": "underlying_symbol"})
    itm_counts = base_symbols.merge(per_symbol_itm.to_pandas(), on="underlying_symbol", how="left").fillna(0)
    count_fields = ["before_itm_filter", "kept_itm", "removed_otm_or_atm", "unknown"]
    itm_counts[count_fields] = itm_counts[count_fields].astype(int)
    totals = itm_counts[count_fields].sum().to_dict()
    kept_rights = (option_itm_audit.filter(pl.col("expiry_status") == "in_the_money")
        .group_by("underlying_symbol").agg(
            (pl.col("option_type") == "call").sum().alias("calls"),
            (pl.col("option_type") == "put").sum().alias("puts")).to_pandas())
    itm_filter_counts = itm_counts.merge(kept_rights, on="underlying_symbol", how="left").rename(
        columns={"underlying_symbol": "symbol", "kept_itm": "remaining"})
    itm_comparison = show_option_filter("In the money at expiration", itm_filter_counts, expiration_comparison)
    display(option_itm_audit.select("underlying_symbol", "contract_symbol", "expiration_session",
        "strike", "split_factor", "expiration_strike", "expiration_spot", "expiry_status").head(20).to_pandas())
    itm_options = option_itm_audit.filter(pl.col("expiry_status") == "in_the_money")
    print("itm_options contains ITM contracts; purchase premiums were not deducted.")

# Percentages use all same-year candidates before the ITM filter, not just survivors.
if not option_itm_audit.is_empty():
    expiry_distribution = (option_itm_audit.with_columns(
        pl.when(pl.col("expiry_status").str.starts_with("unknown")).then(pl.lit("Unknown"))
        .when(pl.col("expiry_status") == "in_the_money").then(pl.lit("ITM"))
        .when(pl.col("expiration_spot") == pl.col("expiration_strike")).then(pl.lit("ATM"))
        .otherwise(pl.lit("OTM")).alias("outcome"))
        .group_by("outcome").len(name="contracts").to_pandas()
        .set_index("outcome").reindex(["ITM", "OTM", "ATM", "Unknown"], fill_value=0))
    expiry_distribution["percent_of_candidates"] = 100 * expiry_distribution["contracts"] / len(option_itm_audit)
    print(f"Expiration outcomes: {EDA_UNIVERSE}, {EDA_YEAR}; denominator = "
          f"{len(option_itm_audit):,} same-year contracts before the ITM filter.")
    display(expiry_distribution.style.format({"contracts": "{:,.0f}", "percent_of_candidates": "{:.2f}%"}))

# Retain original first-day columns for any subsequent filters.
if option_itm_audit.is_empty():
    itm_options = itm_input.head(0)
else:
    survivor_keys = option_itm_audit.filter(pl.col("expiry_status") == "in_the_money").select(
        "underlying_symbol", "contract_symbol")
    itm_options = itm_input.join(survivor_keys, on=["underlying_symbol", "contract_symbol"], how="semi")
filtered_options = itm_options
print(f"Stacked filters: {len(all_options):,} original -> {len(same_year_options):,} "
      f"same-year expirations -> {len(filtered_options):,} ITM survivors.")


In the money at expiration


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,In the money at expiration,113,19,166949,136703,68020,68683,98929,45338,22682,810,601.95,40.74%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,5940,2970,2970,4734,38.55%,2330,640,0
1,BKNG,stored chain,6180,5348,2674,2674,3506,43.27%,2187,487,0
2,NVDA,stored chain,5492,4056,2028,2028,3464,36.93%,1786,242,0
3,META,stored chain,4098,3006,1503,1503,2595,36.68%,1152,351,0
4,TSLA,stored chain,3910,3114,1557,1557,2353,39.82%,493,1064,0
5,AAPL,stored chain,3792,2796,1396,1400,2396,36.81%,808,588,0
6,NFLX,stored chain,3772,2974,1487,1487,2285,39.42%,1026,461,0
7,AMZN,stored chain,3532,2808,1404,1404,2128,39.75%,920,484,0
8,AMD,stored chain,3512,2856,1428,1428,2084,40.66%,1052,376,0
9,BA,stored chain,3468,2776,1388,1388,2080,40.02%,504,884,0


,underlying_symbol,contract_symbol,expiration_session,strike,split_factor,expiration_strike,expiration_spot,expiry_status
0,AAPL,AAPL_put_20240126_245,2024-01-26,245.0,1.0,245.0,192.42,in_the_money
1,AAPL,AAPL240105P00095000,2024-01-05,95.0,1.0,95.0,181.18,out_or_at_the_money
2,AAPL,AAPL240105C00125000,2024-01-05,125.0,1.0,125.0,181.18,in_the_money
3,AAPL,AAPL_call_20240621_100,2024-06-21,100.0,1.0,100.0,207.49,in_the_money
4,AAPL,AAPL_call_20240621_195,2024-06-21,195.0,1.0,195.0,207.49,in_the_money
5,AAPL,AAPL_put_20240517_60,2024-05-17,60.0,1.0,60.0,189.87,out_or_at_the_money
6,AAPL,AAPL240216C00230000,2024-02-16,230.0,1.0,230.0,182.31,out_or_at_the_money
7,AAPL,AAPL240216C00265000,2024-02-16,265.0,1.0,265.0,182.31,out_or_at_the_money
8,AAPL,AAPL240315P00185000,2024-03-15,185.0,1.0,185.0,172.62,in_the_money
9,AAPL,AAPL_call_20240105_250,2024-01-05,250.0,1.0,250.0,181.18,out_or_at_the_money


itm_options contains ITM contracts; purchase premiums were not deducted.
Expiration outcomes: 100B, 2024; denominator = 136,703 same-year contracts before the ITM filter.


,contracts,percent_of_candidates
outcome,,
ITM,"68,020",49.76%
OTM,"67,847",49.63%
ATM,26,0.02%
Unknown,810,0.59%


Stacked filters: 166,949 original -> 136,703 same-year expirations -> 68,020 ITM survivors.


In [6]:
# Stage 2: positive expiration moneyness, following the same-year filter.
# ITM is the chosen definition here; purchase premiums are not deducted.
survivor_keys = itm_options.select("underlying_symbol", "contract_symbol").unique()
moneyness = option_itm_audit.join(
    survivor_keys, on=["underlying_symbol", "contract_symbol"], how="semi").to_pandas()
moneyness["moneyness_pct"] = 100 * (moneyness["expiration_spot"] / moneyness["expiration_strike"] - 1)
moneyness.loc[moneyness["option_type"] == "put", "moneyness_pct"] *= -1
display(moneyness["moneyness_pct"].describe())
# Explicit previous-stage output; no percentile threshold is applied.
moneyness_options = itm_options
moneyness_comparison = itm_comparison
filtered_options = moneyness_options


count    68020.000000
mean        58.246189
std        330.125441
min          0.002105
25%          9.618421
50%         22.147758
75%         52.800431
max      26840.000000
Name: moneyness_pct, dtype: float64

## Option filtering — Positive ask-to-bid profit after positive moneyness

The filter order is **same-year expiration → positive expiration moneyness →
positive ask-to-bid profit**. This cell consumes only `moneyness_options` and
retains contracts with a finite `profit_pct > 0`. There is no percentile filter.
Buy at the selected first-day ask and sell at the last stored bid after entry
and on or before expiration. Break-even returns, losses, and unpriced contracts
are removed; unknowns are reported separately in the same comparison table.
`profitable_options` and `filtered_options` hold the final survivors.

`profit_pct = 100 × (exit_bid_per_original_unit / entry_ask − 1)`

Forward splits adjust contract counts/strikes using the same mapping as the
trainer. Contract multipliers cancel in the percentage. Fees are excluded, but
the ask-to-bid spread is included. Zero exit bids represent a 100% loss.

The final stored quote may precede expiration; `days_before_expiration` and
`exit_on_expiration_session` expose that gap. Missing/invalid quotes and unsupported
split adjustments are reported separately, not assigned a zero return. The last
row is used even if its bid is invalid; we do not search backward for a better bid.
These are quote-based hypothetical returns, not proof a full-size order could
fill. Because the input was selected using expiration ITM outcomes, this is a
hindsight-selected EDA sample, not an unbiased strategy backtest.

In [7]:
from datetime import timedelta
# Also supports rerunning this cell in a kernel with the older setup cell loaded.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from quant_orchestrator.research_tools.frozen_option_adjustments import split_adjusted_members

# Stage 3 consumes only positive-moneyness survivors, even when rerun.
profit_input = moneyness_options
return_audits = []
for (symbol,), selected in profit_input.group_by("underlying_symbol", maintain_order=True):
    expiry_days = option_itm_audit.select("underlying_symbol", "contract_symbol", "expiration_session")
    entries = selected.select("underlying_symbol", "contract_symbol", "expiration", "strike", "option_type",
                              pl.col("ask").alias("entry_ask")).join(
        expiry_days, on=["underlying_symbol", "contract_symbol"], how="left")
    stop = entries["expiration_session"].max()
    quotes = read_thetadata_eod_option_chain(
        symbol, start_date=first_day, end_date=stop, backend=warehouse.backend,
        columns=["snapshot_date", "contract_symbol", "expiration", "strike", "option_type", "bid", "ask"])
    members = entries.select("contract_symbol", "expiration", "strike", "option_type").with_columns(
        pl.col("contract_symbol").alias("document_symbol"), pl.lit(1.).alias("weight"))
    splits = warehouse.read_fundamentals(symbol, section="historical_splits",
        start=first_day.date().isoformat(), end=stop.date().isoformat())
    split_rows = [r for r in splits.sort("date").to_dicts() if first_day < r["date"] <= stop] if splits.height else []
    boundaries = sorted({first_day, *[r["date"] for r in split_rows], stop + timedelta(days=1)})
    pieces, unsupported_from = [], None
    for left, right in zip(boundaries, boundaries[1:]):
        factor = 1.
        for split in split_rows:
            if split["date"] <= left:
                numerator, denominator = split.get("numerator"), split.get("denominator")
                ratio = numerator / denominator if numerator and denominator and denominator > 0 else float("nan")
                if not (1 <= ratio < float("inf")):
                    unsupported_from = split["date"] if unsupported_from is None else min(unsupported_from, split["date"])
                else:
                    factor *= ratio
        if unsupported_from is not None:
            break
        if quotes.is_empty():
            continue
        segment = quotes.filter((pl.col("snapshot_date") >= left) & (pl.col("snapshot_date") < right))
        if segment.is_empty():
            continue
        adjusted = split_adjusted_members(members, segment, factor)
        observed = segment.join(adjusted.select("contract_symbol", "document_symbol", "weight"),
                                on="contract_symbol", how="inner")
        pieces.append(observed.select(pl.col("document_symbol").alias("contract_symbol"),
            "snapshot_date", pl.col("bid").alias("exit_bid"), pl.col("ask").alias("observed_ask"),
            pl.col("weight").alias("exit_contract_factor")))
    sessions_in_window = calendar.sessions_in_range(first_day, stop).tz_localize(None).to_pydatetime().tolist()
    expected_rows = [{"expiration_session": day, "expected_quote_days": len(calendar.sessions_in_range(first_day, day))}
                     for day in entries["expiration_session"].unique().to_list()]
    expected_days = pl.DataFrame(expected_rows).with_columns(
        pl.col("expiration_session").cast(entries.schema["expiration_session"]))
    observation_counts = pl.DataFrame(schema={"contract_symbol": pl.String, "valid_quote_days": pl.UInt32})
    if pieces:
        paths = pl.concat(pieces, how="diagonal_relaxed").join(
            entries.select("contract_symbol", "expiration_session"), on="contract_symbol", how="inner")
        valid_quotes = paths.filter(
            pl.col("snapshot_date").is_in(sessions_in_window) &
            (pl.col("snapshot_date") <= pl.col("expiration_session")) &
            pl.col("exit_bid").is_finite() & pl.col("observed_ask").is_finite() &
            (pl.col("exit_bid") > 0) & (pl.col("observed_ask") > 0) &
            (pl.col("observed_ask") >= pl.col("exit_bid")))
        observation_counts = valid_quotes.group_by("contract_symbol").agg(
            pl.col("snapshot_date").n_unique().alias("valid_quote_days"))
        paths = paths.filter((pl.col("snapshot_date") > first_day) &
                             (pl.col("snapshot_date") <= pl.col("expiration_session")))
        last = (paths.sort("snapshot_date").group_by("contract_symbol").agg(
            pl.col("snapshot_date").last().alias("exit_date"),
            pl.col("exit_bid").last(), pl.col("exit_contract_factor").last()))
    else:
        last = pl.DataFrame(schema={"contract_symbol": pl.String, "exit_date": pl.Datetime("ns"),
                                    "exit_bid": pl.Float64, "exit_contract_factor": pl.Float64})
    result = (entries.join(last, on="contract_symbol", how="left")
        .join(observation_counts, on="contract_symbol", how="left")
        .join(expected_days, on="expiration_session", how="left")
        .with_columns(pl.col("valid_quote_days").fill_null(0))
        .with_columns((pl.col("valid_quote_days") / pl.col("expected_quote_days")).alias("quote_coverage")))
    result = result.with_columns(
        pl.lit(first_day).alias("entry_date"),
        (pl.col("exit_bid") * pl.col("exit_contract_factor")).alias("exit_bid_per_original_unit"))
    unsupported = pl.col("expiration_session") >= unsupported_from if unsupported_from is not None else pl.lit(False)
    result = result.with_columns(
        pl.when(unsupported).then(pl.lit("unsupported_split"))
        .when(~(pl.col("entry_ask").is_finite() & (pl.col("entry_ask") > 0)).fill_null(False)).then(pl.lit("invalid_entry_ask"))
        .when(pl.col("exit_date").is_null()).then(pl.lit("missing_later_quote"))
        .when(~(pl.col("exit_bid").is_finite() & (pl.col("exit_bid") >= 0)).fill_null(False)).then(pl.lit("invalid_exit_bid"))
        .otherwise(pl.lit("priced")).alias("return_status"))
    result = result.with_columns(
        pl.when(pl.col("return_status") == "priced")
        .then(100 * (pl.col("exit_bid_per_original_unit") / pl.col("entry_ask") - 1))
        .otherwise(None).alias("profit_pct"),
        (pl.col("expiration_session") - pl.col("exit_date")).dt.total_days().alias("days_before_expiration"),
        (pl.col("exit_date") == pl.col("expiration_session")).fill_null(False).alias("exit_on_expiration_session"))
    return_audits.append(result)

option_return_audit = pl.concat(return_audits, how="diagonal_relaxed") if return_audits else pl.DataFrame()
if option_return_audit.is_empty():
    print("No filtered options to price. Run the stacked EDA cells first.")
else:
    option_returns = option_return_audit.to_pandas()
    print("Profit distribution before the positive-profit filter:")
    display(option_returns["profit_pct"].describe())
    display(option_returns.groupby("return_status").size().rename("contracts").to_frame())
    priced = option_returns.loc[option_returns.return_status == "priced"]
    display(pd.DataFrame([{
        "universe": EDA_UNIVERSE, "year": EDA_YEAR,
        "filtered_contracts": len(profit_input), "priced_contracts": len(priced),
        "positive_return_pct": 100 * priced.profit_pct.gt(0).mean(),
        "exit_on_expiration_session": int(priced.exit_on_expiration_session.sum()),
        "exit_before_expiration_session": int((~priced.exit_on_expiration_session).sum()),
    }]))
    display(option_returns[["underlying_symbol", "contract_symbol", "entry_date", "entry_ask",
        "exit_date", "exit_bid", "exit_contract_factor", "profit_pct", "days_before_expiration", "return_status"]].head(20))
# Keep strictly positive, priced returns; break-even/losses and unknowns do not survive.
if option_return_audit.is_empty():
    profitable_options = profit_input.head(0)
    profit_filter_counts = pd.DataFrame(columns=["symbol", "remaining", "calls", "puts", "unknown"])
else:
    positive = ((pl.col("return_status") == "priced") &
                pl.col("profit_pct").is_finite() & (pl.col("profit_pct") > 0)).fill_null(False)
    profit_filter_counts = option_return_audit.group_by("underlying_symbol").agg(
        positive.sum().alias("remaining"),
        (positive & (pl.col("option_type") == "call")).sum().alias("calls"),
        (positive & (pl.col("option_type") == "put")).sum().alias("puts"),
        (pl.col("return_status") != "priced").sum().alias("unknown")
    ).rename({"underlying_symbol": "symbol"}).to_pandas()
    profit_survivor_keys = option_return_audit.filter(positive).select("underlying_symbol", "contract_symbol")
    profitable_options = profit_input.join(profit_survivor_keys,
        on=["underlying_symbol", "contract_symbol"], how="semi")
    print("Profit distribution after keeping only positive returns:")
    display(option_return_audit.filter(positive)["profit_pct"].to_pandas().describe())
filtered_options = profitable_options
profit_comparison = show_option_filter("Positive ask-to-bid profit", profit_filter_counts, moneyness_comparison)
print(f"Stacked survivors: {len(all_options):,} original -> {len(same_year_options):,} same-year "
      f"-> {len(moneyness_options):,} positive moneyness -> {len(filtered_options):,} positive profit.")


Profit distribution before the positive-profit filter:


count    66124.000000
mean       125.229084
std       1011.490542
min       -100.000000
25%        -20.000000
50%          6.359015
75%         57.484779
max      61062.790698
Name: profit_pct, dtype: float64

,contracts
return_status,
missing_later_quote,1896
priced,66124


,universe,year,filtered_contracts,priced_contracts,positive_return_pct,exit_on_expiration_session,exit_before_expiration_session
0,100B,2024,68020,66124,56.448491,61483,4641


,underlying_symbol,contract_symbol,entry_date,entry_ask,exit_date,exit_bid,exit_contract_factor,profit_pct,days_before_expiration,return_status
0,AAPL,AAPL_put_20240126_245,2024-01-02,60.15,2024-01-26,51.90,1.0,-13.715711,0.0,priced
1,AAPL,AAPL240105C00125000,2024-01-02,60.95,2024-01-05,54.95,1.0,-9.844135,0.0,priced
2,AAPL,AAPL_call_20240621_100,2024-01-02,88.40,2024-06-18,112.85,1.0,27.658371,3.0,priced
3,AAPL,AAPL_call_20240621_195,2024-01-02,8.70,2024-06-18,19.20,1.0,120.689655,3.0,priced
4,AAPL,AAPL240315P00185000,2024-01-02,6.15,2024-03-15,12.35,1.0,100.813008,0.0,priced
5,AAPL,AAPL_put_20240315_215,2024-01-02,30.65,2024-03-15,40.95,1.0,33.605220,0.0,priced
6,AAPL,AAPL240216C00085000,2024-01-02,101.70,2024-02-16,96.80,1.0,-4.818092,0.0,priced
7,AAPL,AAPL_put_20240621_285,2024-01-02,99.65,2024-06-18,69.30,1.0,-30.456598,3.0,priced
8,AAPL,AAPL240126C00190000,2024-01-02,1.93,2024-01-26,2.06,1.0,6.735751,0.0,priced
9,AAPL,AAPL_call_20240105_100,2024-01-02,86.25,2024-01-04,80.65,1.0,-6.492754,1.0,priced


...

count    37326.000000
mean       247.996020
std       1333.146826
min          0.013980
25%         16.763958
50%         46.815344
75%        133.731437
max      61062.790698
Name: profit_pct, dtype: float64

Positive ask-to-bid profit


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,Positive ask-to-bid profit,113,19,166949,68020,37326,30694,129623,29380,7946,1896,330.32,22.36%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,2970,2190,780,5514,28.43%,2032,158,0
1,BKNG,stored chain,6180,2674,1520,1154,4660,24.60%,1389,131,0
2,NVDA,stored chain,5492,2028,1781,247,3711,32.43%,1781,0,4
3,META,stored chain,4098,1503,1140,363,2958,27.82%,1140,0,2
4,TSLA,stored chain,3910,1557,997,560,2913,25.50%,0,997,13
5,AAPL,stored chain,3792,1396,645,751,3147,17.01%,463,182,0
6,NFLX,stored chain,3772,1487,1004,483,2768,26.62%,1004,0,0
7,AMZN,stored chain,3532,1404,896,508,2636,25.37%,852,44,0
8,AMD,stored chain,3512,1428,894,534,2618,25.46%,894,0,86
9,BA,stored chain,3468,1388,739,649,2729,21.31%,0,739,235


Stacked survivors: 166,949 original -> 136,703 same-year -> 68,020 positive moneyness -> 37,326 positive profit.


## Option filtering — Minimum history and quote coverage

Next, filter the **positive-profit survivors** to require both:

- At least **20 distinct valid daily quote observations**.
- Quotes on at least **80% of NYSE trading sessions**, counting entry through the
  expiration trading session, inclusive. Missing tail days remain in the denominator.

A valid observation has finite bid/ask values, `bid > 0`, `ask > 0`, and
`ask >= bid`. Zero volume is allowed. Duplicate dates count once; no observations
are forward-filled. Forward-split mappings are reused from the profit calculation.
The preceding cell collects these counts during its existing quote scan, so this
cell does not download or reread the history. Rerun the profit cell once after
updating the notebook to populate the added coverage fields.

Required observations are `max(20, ceil(0.80 × expected_trading_days))`.
A 10-session contract cannot pass; a 25-session contract needs 20 valid days,
and a 60-session contract needs 48. Change the parameters below to compare
thresholds. These are starting EDA thresholds, not validated training defaults.

The order is now **same-year → positive moneyness → positive profit → history
coverage**. `history_options` and `filtered_options` contain the final survivors.
The same comparison table and original-symbol denominator are retained.

In [8]:
MIN_OBSERVATIONS = 20
MIN_COVERAGE = 0.80
if MIN_OBSERVATIONS < 1 or not 0 < MIN_COVERAGE <= 1:
    raise ValueError("Require a positive observation minimum and coverage in (0, 1].")

history_input = profitable_options
if history_input.is_empty():
    history_options = history_input.head(0)
    history_filter_counts = pd.DataFrame(columns=["symbol", "remaining", "calls", "puts", "unknown"])
    option_history_audit = pl.DataFrame()
else:
    required_fields = {"valid_quote_days", "expected_quote_days", "quote_coverage"}
    if not required_fields <= set(option_return_audit.columns):
        raise RuntimeError("Rerun the updated profit cell first to collect daily quote coverage.")
    option_history_audit = option_return_audit.join(
        history_input.select("underlying_symbol", "contract_symbol"),
        on=["underlying_symbol", "contract_symbol"], how="semi")
    option_history_audit = option_history_audit.with_columns(
        pl.max_horizontal(pl.lit(MIN_OBSERVATIONS),
            (pl.col("expected_quote_days") * MIN_COVERAGE).ceil()).cast(pl.Int64).alias("required_quote_days"))
    option_history_audit = option_history_audit.with_columns(
        ((pl.col("valid_quote_days") >= pl.col("required_quote_days")) &
         (pl.col("expected_quote_days") > 0)).fill_null(False).alias("keep_history"))
    history_keys = option_history_audit.filter(pl.col("keep_history")).select("underlying_symbol", "contract_symbol")
    history_options = history_input.join(history_keys, on=["underlying_symbol", "contract_symbol"], how="semi")
    history_filter_counts = option_history_audit.group_by("underlying_symbol").agg(
        pl.col("keep_history").sum().alias("remaining"),
        (pl.col("keep_history") & (pl.col("option_type") == "call")).sum().alias("calls"),
        (pl.col("keep_history") & (pl.col("option_type") == "put")).sum().alias("puts"),
        (pl.col("valid_quote_days").is_null() | pl.col("expected_quote_days").is_null()).sum().alias("unknown")
    ).rename({"underlying_symbol": "symbol"}).to_pandas()

filtered_options = history_options
history_comparison = show_option_filter(
    f"At least {MIN_OBSERVATIONS} quote days and {MIN_COVERAGE:.0%} coverage",
    history_filter_counts, profit_comparison)
if not option_history_audit.is_empty():
    display(option_history_audit.select("underlying_symbol", "contract_symbol", "expected_quote_days",
        "valid_quote_days", "required_quote_days", "quote_coverage", "keep_history").head(20).to_pandas())
    display(option_history_audit.select("valid_quote_days", "expected_quote_days", "quote_coverage").to_pandas().describe())
print(f"History filter: {len(history_input):,} positive-profit candidates -> {len(filtered_options):,} survivors.")

At least 20 quote days and 80% coverage


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,At least 20 quote days and 80% coverage,113,19,166949,37326,20993,16333,145956,17860,3133,0,185.78,12.57%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,2190,725,1465,6979,9.41%,725,0,0
1,BKNG,stored chain,6180,1520,1019,501,5161,16.49%,978,41,0
2,NVDA,stored chain,5492,1781,1232,549,4260,22.43%,1232,0,0
3,META,stored chain,4098,1140,729,411,3369,17.79%,729,0,0
4,TSLA,stored chain,3910,997,551,446,3359,14.09%,0,551,0
5,AAPL,stored chain,3792,645,241,404,3551,6.36%,163,78,0
6,NFLX,stored chain,3772,1004,706,298,3066,18.72%,706,0,0
7,AMZN,stored chain,3532,896,293,603,3239,8.30%,293,0,0
8,AMD,stored chain,3512,894,240,654,3272,6.83%,240,0,0
9,BA,stored chain,3468,739,305,434,3163,8.79%,0,305,0


,underlying_symbol,contract_symbol,expected_quote_days,valid_quote_days,required_quote_days,quote_coverage,keep_history
0,AAPL,AAPL_call_20240621_100,119,67,96,0.563025,False
1,AAPL,AAPL_call_20240621_195,119,67,96,0.563025,False
2,AAPL,AAPL240315P00185000,52,52,42,1.000000,True
3,AAPL,AAPL_put_20240315_215,52,30,42,0.576923,False
4,AAPL,AAPL240126C00190000,18,18,20,1.000000,False
5,AAPL,AAPL241220C00065000,246,246,197,1.000000,True
6,AAPL,AAPL240126C00050000,18,18,20,1.000000,False
7,AAPL,AAPL240126C00185000,18,18,20,1.000000,False
8,AAPL,AAPL240719C00140000,138,138,111,1.000000,True
9,AAPL,AAPL_call_20241220_130,246,140,197,0.569106,False


,valid_quote_days,expected_quote_days,quote_coverage
count,37326.000000,37326.000000,37326.000000
mean,54.538900,59.112576,0.929327
std,57.106331,58.573092,0.207715
min,2.000000,4.000000,0.031250
25%,13.000000,13.000000,1.000000
50%,28.000000,28.000000,1.000000
75%,96.000000,119.000000,1.000000
max,246.000000,246.000000,1.000000


History filter: 37,326 positive-profit candidates -> 20,993 survivors.


## Option filtering — Profit percentile separately for calls and puts

After history coverage, keep contracts at or above the **50th profit percentile
within their own option type**. Calls are compared with calls; puts with puts.
Each cutoff uses that type's survivors across the entire selected universe,
not a separate cutoff per symbol. Ties at the cutoff are retained.

This keeps roughly the top half of each side. It does not force equal call/put
counts because their input populations may differ. The cutoff table shows each
side's candidate count, threshold, and retained count.

Order: same-year → positive moneyness → positive profit → valid history →
profit percentile by call/put. `top_profit_options` and `filtered_options` hold
final survivors. Returns include spreads and split adjustments, but exclude fees.
This is retrospective EDA.

In [9]:
PROFIT_QUANTILE = 0.50
if not 0 <= PROFIT_QUANTILE <= 1:
    raise ValueError("PROFIT_QUANTILE must be between 0 and 1.")

percentile_input = history_options
if percentile_input.is_empty():
    profit_cutoffs = {}
    top_profit_options = percentile_input.head(0)
    percentile_filter_counts = pd.DataFrame(columns=["symbol", "remaining", "calls", "puts", "unknown"])
    option_profit_percentile_audit = pl.DataFrame()
    print("No history-filter survivors; no profit percentile can be computed.")
else:
    option_profit_percentile_audit = option_return_audit.join(
        percentile_input.select("underlying_symbol", "contract_symbol"),
        on=["underlying_symbol", "contract_symbol"], how="semi")
    eligible_profit = ((pl.col("return_status") == "priced") &
                      pl.col("profit_pct").is_finite() & (pl.col("profit_pct") > 0)).fill_null(False)
    eligible = option_profit_percentile_audit.filter(eligible_profit).to_pandas()
    cutoff_table = (eligible.groupby("option_type")["profit_pct"].quantile(PROFIT_QUANTILE)
                    .rename("profit_cutoff_pct").reset_index())
    profit_cutoffs = dict(zip(cutoff_table["option_type"], cutoff_table["profit_cutoff_pct"]))
    cutoffs = pl.DataFrame({"option_type": list(profit_cutoffs), "profit_cutoff_pct": list(profit_cutoffs.values())},
                          schema={"option_type": pl.String, "profit_cutoff_pct": pl.Float64})
    option_profit_percentile_audit = option_profit_percentile_audit.join(cutoffs, on="option_type", how="left")
    passes = eligible_profit & (pl.col("profit_pct") >= pl.col("profit_cutoff_pct"))
    option_profit_percentile_audit = option_profit_percentile_audit.with_columns(
        passes.fill_null(False).alias("keep_profit_percentile"))
    kept_keys = option_profit_percentile_audit.filter(pl.col("keep_profit_percentile")).select(
        "underlying_symbol", "contract_symbol")
    top_profit_options = percentile_input.join(kept_keys, on=["underlying_symbol", "contract_symbol"], how="semi")
    percentile_filter_counts = option_profit_percentile_audit.group_by("underlying_symbol").agg(
        pl.col("keep_profit_percentile").sum().alias("remaining"),
        (pl.col("keep_profit_percentile") & (pl.col("option_type") == "call")).sum().alias("calls"),
        (pl.col("keep_profit_percentile") & (pl.col("option_type") == "put")).sum().alias("puts"),
        (~eligible_profit).sum().alias("unknown")
    ).rename({"underlying_symbol": "symbol"}).to_pandas()
    before_by_type = eligible.groupby("option_type").size().rename("candidates")
    kept_by_type = (option_profit_percentile_audit.filter(pl.col("keep_profit_percentile"))
                   .group_by("option_type").len(name="retained").to_pandas())
    cutoff_summary = cutoff_table.merge(before_by_type, on="option_type").merge(kept_by_type, on="option_type", how="left")
    cutoff_summary["retained"] = cutoff_summary["retained"].fillna(0).astype(int)
    print(f"{PROFIT_QUANTILE:.0%} profit percentile, separately for calls and puts:")
    display(cutoff_summary)
    display(option_profit_percentile_audit.filter(pl.col("keep_profit_percentile"))["profit_pct"].to_pandas().describe())

filtered_options = top_profit_options
profit_percentile_comparison = show_option_filter(
    f"Profit percentile {100 * PROFIT_QUANTILE:g} within call/put", percentile_filter_counts, history_comparison)
print(f"Profit percentile filter: {len(percentile_input):,} -> {len(filtered_options):,} contracts.")

50% profit percentile, separately for calls and puts:


,option_type,profit_cutoff_pct,candidates,retained
0,call,65.073044,17860,8930
1,put,34.622144,3133,1567


count    10497.000000
mean       610.629766
std       2388.651683
min         34.622144
25%         92.892157
50%        163.176265
75%        359.770115
max      61062.790698
Name: profit_pct, dtype: float64

Profit percentile 50 within call/put


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,Profit percentile 50 within call/put,113,19,166949,20993,10497,10496,156452,8930,1567,0,92.89,6.29%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,725,506,219,7198,6.57%,506,0,0
1,BKNG,stored chain,6180,1019,99,920,6081,1.60%,99,0,0
2,NVDA,stored chain,5492,1232,1186,46,4306,21.60%,1186,0,0
3,META,stored chain,4098,729,586,143,3512,14.30%,586,0,0
4,TSLA,stored chain,3910,551,380,171,3530,9.72%,0,380,0
5,AAPL,stored chain,3792,241,83,158,3709,2.19%,63,20,0
6,NFLX,stored chain,3772,706,458,248,3314,12.14%,458,0,0
7,AMZN,stored chain,3532,293,140,153,3392,3.96%,140,0,0
8,AMD,stored chain,3512,240,120,120,3392,3.42%,120,0,0
9,BA,stored chain,3468,305,304,1,3164,8.77%,0,304,0


Profit percentile filter: 20,993 -> 10,497 contracts.


## Sample up to 5 calls and 5 puts per symbol

After the **50th-percentile profit filter applied separately to calls and puts**,
randomly sample **up to 5 calls and up to 5 puts per underlying symbol**, without
replacement. A side with fewer than 5 survivors keeps all of them. Missing calls
are not replaced with extra puts, or vice versa.

`SAMPLE_SEED` makes the selection repeatable. Each symbol/option-type group uses
its own seeded generator and sorted contract identifiers, so input ordering and
other groups do not change its sample. Change the seed for another selection.
`sampled_options` and `filtered_options` hold the sample; the following plot uses it.

In [10]:
import random

OPTIONS_PER_SIDE = 5
SAMPLE_SEED = 0

def sample_options_per_side(options, count, seed):
    if not isinstance(count, int) or isinstance(count, bool) or count < 1:
        raise ValueError("OPTIONS_PER_SIDE must be a positive integer.")
    if options.is_empty():
        return options.clone()
    groups = []
    for (symbol, option_type), group in options.group_by(["underlying_symbol", "option_type"], maintain_order=True):
        group = group.sort("contract_symbol")
        rng = random.Random(f"{seed}:{symbol}:{option_type}")
        chosen = rng.sample(range(len(group)), min(count, len(group)))
        groups.append(group[chosen])
    return pl.concat(groups, how="diagonal_relaxed").sort("underlying_symbol", "contract_symbol")

sampling_input = top_profit_options
sampled_options = sample_options_per_side(sampling_input, OPTIONS_PER_SIDE, SAMPLE_SEED)
filtered_options = sampled_options
sampling_filter_counts = sampled_options.group_by("underlying_symbol").agg(
    pl.len().alias("remaining"),
    (pl.col("option_type") == "call").sum().alias("calls"),
    (pl.col("option_type") == "put").sum().alias("puts"),
    pl.lit(0).alias("unknown")
).rename({"underlying_symbol": "symbol"}).to_pandas()
sampling_comparison = show_option_filter(
    f"Random sample: up to {OPTIONS_PER_SIDE} calls and puts each per symbol (seed {SAMPLE_SEED})",
    sampling_filter_counts, profit_percentile_comparison)
print(f"Sample: {len(sampling_input):,} candidates -> {len(filtered_options):,} contracts "
      f"across {filtered_options['underlying_symbol'].n_unique()} symbols.")

Random sample: up to 5 calls and puts each per symbol (seed 0)


,stage,symbols_with_chain,symbols_missing_chain,original,before,remaining,removed_this_step,removed_total,calls,puts,unknown,average_remaining,retained_pct_original
0,Random sample: up to 5 calls and puts each per symbol (seed 0),113,19,166949,10497,545,9952,166404,399,146,0,4.82,0.33%


,symbol,status,original,before,remaining,removed_this_step,removed_total,retained_pct_original,calls,puts,unknown
0,AVGO,stored chain,7704,506,5,501,7699,0.06%,5,0,0
1,BKNG,stored chain,6180,99,5,94,6175,0.08%,5,0,0
2,NVDA,stored chain,5492,1186,5,1181,5487,0.09%,5,0,0
3,META,stored chain,4098,586,5,581,4093,0.12%,5,0,0
4,TSLA,stored chain,3910,380,5,375,3905,0.13%,0,5,0
5,AAPL,stored chain,3792,83,10,73,3782,0.26%,5,5,0
6,NFLX,stored chain,3772,458,5,453,3767,0.13%,5,0,0
7,AMZN,stored chain,3532,140,5,135,3527,0.14%,5,0,0
8,AMD,stored chain,3512,120,5,115,3507,0.14%,5,0,0
9,BA,stored chain,3468,304,5,299,3463,0.14%,0,5,0


Sample: 10,497 candidates -> 545 contracts across 103 symbols.


## Plot the final surviving option time series

This cell plots **only `filtered_options`**, after every preceding filter.
Select `PLOT_SYMBOL` below; `None` uses the first symbol with survivors. All surviving
contracts for that symbol are shown, with separate call and put panels. Change
`PRICE_FIELD` to `"bid"`, `"ask"`, or `"mid"`. Plotly supports zoom and hover, and
clicking legend entries hides or isolates contracts.

Prices are expressed per original option unit: forward-split contract counts are
applied so a split does not create an artificial drop. The optional entry-ask
normalization sets that purchase cost to 100. Missing/invalid quote days appear
as gaps and are not forward-filled. No new contracts are added and the final
filtered set is unchanged. The complete plotted data is in `survivor_plot_data`.
The chart uses the installed `plotly` package.

In [ ]:
import plotly.express as px

PLOT_SYMBOL = "AAPL"       # Change to another surviving symbol, or None for the first.
PRICE_FIELD = "mid"         # "mid", "bid", or "ask"
NORMALIZE_TO_ENTRY_ASK = False

if PRICE_FIELD not in {"mid", "bid", "ask"}:
    raise ValueError("PRICE_FIELD must be mid, bid, or ask.")
available_plot_symbols = sorted(filtered_options["underlying_symbol"].unique().to_list())
print("Symbols with survivors:", ", ".join(available_plot_symbols))
survivor_plot_data = pd.DataFrame()
if not available_plot_symbols:
    print("No surviving options to plot; run the filtering cells first.")
else:
    plot_symbol = PLOT_SYMBOL or available_plot_symbols[0]
    if plot_symbol not in available_plot_symbols:
        raise ValueError(f"{plot_symbol} has no survivors. Choose from available_plot_symbols above.")
    selected_plot = filtered_options.filter(pl.col("underlying_symbol") == plot_symbol)
    plot_entries = selected_plot.select("contract_symbol", "expiration", "strike", "option_type",
        pl.col("ask").alias("entry_ask")).join(
        option_itm_audit.filter(pl.col("underlying_symbol") == plot_symbol)
            .select("contract_symbol", "expiration_session"), on="contract_symbol", how="left")
    plot_stop = plot_entries["expiration_session"].max()
    plot_quotes = read_thetadata_eod_option_chain(plot_symbol, start_date=first_day, end_date=plot_stop,
        columns=["snapshot_date", "contract_symbol", "expiration", "strike", "option_type", "bid", "ask"],
        backend=warehouse.backend)
    plot_members = plot_entries.select("contract_symbol", "expiration", "strike", "option_type").with_columns(
        pl.col("contract_symbol").alias("document_symbol"), pl.lit(1.).alias("weight"))
    plot_splits = warehouse.read_fundamentals(plot_symbol, section="historical_splits",
        start=first_day.date().isoformat(), end=plot_stop.date().isoformat())
    plot_split_rows = [r for r in plot_splits.sort("date").to_dicts() if first_day < r["date"] <= plot_stop] if plot_splits.height else []
    plot_boundaries = sorted({first_day, *[r["date"] for r in plot_split_rows], plot_stop + timedelta(days=1)})
    plot_parts = []
    for left, right in zip(plot_boundaries, plot_boundaries[1:]):
        factor = 1.
        for split in plot_split_rows:
            if split["date"] <= left:
                numerator, denominator = split.get("numerator"), split.get("denominator")
                ratio = numerator / denominator if numerator and denominator and denominator > 0 else float("nan")
                if not 1 <= ratio < float("inf"):
                    raise ValueError("Plotting this split requires an explicit deliverable mapping.")
                factor *= ratio
        segment = plot_quotes.filter((pl.col("snapshot_date") >= left) & (pl.col("snapshot_date") < right))
        if segment.is_empty():
            continue
        mapped = split_adjusted_members(plot_members, segment, factor)
        part = segment.join(mapped.select("contract_symbol", "document_symbol", "weight"),
                            on="contract_symbol", how="inner")
        quote = ((pl.col("bid") + pl.col("ask")) / 2 if PRICE_FIELD == "mid" else pl.col(PRICE_FIELD))
        valid = (pl.col("bid").is_finite() & pl.col("ask").is_finite() &
                 (pl.col("bid") > 0) & (pl.col("ask") > 0) & (pl.col("ask") >= pl.col("bid")))
        plot_parts.append(part.select(pl.col("document_symbol").alias("contract_symbol"),
            pl.col("snapshot_date").alias("date"),
            pl.when(valid).then(quote * pl.col("weight")).otherwise(None).alias("price")))
    if not plot_parts:
        print("No price observations found for the selected survivors.")
    else:
        observations = pl.concat(plot_parts, how="diagonal_relaxed").unique(["contract_symbol", "date"])
        sessions = calendar.sessions_in_range(first_day, plot_stop).tz_localize(None).to_pydatetime().tolist()
        grid = plot_entries.join(pl.DataFrame({"date": sessions}), how="cross").filter(
            pl.col("date") <= pl.col("expiration_session"))
        observations = observations.with_columns(pl.col("date").cast(grid.schema["date"]))
        plot_frame = grid.join(observations, on=["contract_symbol", "date"], how="left").sort("contract_symbol", "date")
        if NORMALIZE_TO_ENTRY_ASK:
            plot_frame = plot_frame.with_columns((100 * pl.col("price") / pl.col("entry_ask")).alias("price"))
        survivor_plot_data = plot_frame.to_pandas()
        y_label = "Value / entry ask × 100" if NORMALIZE_TO_ENTRY_ASK else f"{PRICE_FIELD.title()} per original option unit ($)"
        fig = px.line(survivor_plot_data, x="date", y="price", color="contract_symbol", facet_row="option_type",
            hover_data=["strike", "expiration", "entry_ask"],
            labels={"price": y_label, "date": "Trading date", "contract_symbol": "Contract"},
            title=f"{plot_symbol}: {len(selected_plot):,} surviving option contracts — {EDA_YEAR}",
            template="plotly_white", height=700)
        fig.update_traces(connectgaps=False)
        fig.update_layout(hovermode="closest", legend_title_text="Surviving contracts")
        fig.show()


Symbols with survivors: AAPL, ABBV, ADI, AMAT, AMD, AMGN, AMZN, ANET, APH, APP, ASML, AVGO, AXP, BA, BAC, BKNG, BLK, BMY, BX, C, CAT, CDNS, COP, COST, CRM, CRWD, CVS, DE, DELL, DIS, DUK, FTNT, GD, GE, GILD, GLW, GOOG, GOOGL, GS, HD, HOOD, HWM, IBKR, IBM, INTC, ISRG, JNJ, JPM, KLAC, KO, LLY, LMT, LOW, LRCX, MA, MCD, META, MO, MRK, MRVL, MS, MSFT, MU, NEE, NEM, NFLX, NOW, NVDA, ORCL, PANW, PEP, PFE, PG, PGR, PH, PLD, PLTR, PM, PNC, PWR, QCOM, RTX, SBUX, SCCO, SCHW, SO, SYK, T, TJX, TMUS, TSLA, TXN, UBER, UNH, UNP, V, VRT, VRTX, VZ, WDC, WFC, WMT, XOM


## Instrument inventory for one issuer

Select `INVENTORY_SYMBOL` to see its equity series and **final sampled option
contracts**, including identifiers, strikes, expirations, and coverage. The summary
keeps bonds visible as **not integrated into this workflow**, rather than claiming
the issuer has no bonds or that no bond data exists anywhere.

The current trainer supports equity and option modalities. Financial statements,
sparse issuer events, Treasury rates, and other macro series are model context,
not additional traded instruments. This inventory describes our EDA selection:
the training launcher still constructs its own frozen synthetic option baskets
and does not yet consume the filtered/sampled individual contracts automatically.

In [ ]:
INVENTORY_SYMBOL = "AAPL"

inventory_symbol = INVENTORY_SYMBOL.strip().upper()
inventory_rows = []
equity_prices = warehouse.read_prices(inventory_symbol, provider="fmp",
    start=first_day.date().isoformat(), end=f"{EDA_YEAR}-12-31")
if not equity_prices.is_empty():
    inventory_rows.append({
        "instrument_type": "Equity", "instrument_id": inventory_symbol,
        "underlying_symbol": inventory_symbol, "strike": None, "expiration": None,
        "valid_quote_days": None, "equity_price_days": equity_prices["date"].n_unique(),
        "source": "FMP equity history", "selection": "equity price series",
    })
selected_inventory = filtered_options.filter(pl.col("underlying_symbol") == inventory_symbol)
if not selected_inventory.is_empty():
    option_details = selected_inventory.select("underlying_symbol", "contract_symbol", "option_type", "strike", "expiration").join(
        option_return_audit.select("underlying_symbol", "contract_symbol", "entry_ask", "profit_pct", "valid_quote_days", "quote_coverage"),
        on=["underlying_symbol", "contract_symbol"], how="left")
    for row in option_details.sort("option_type", "expiration", "strike").to_dicts():
        inventory_rows.append({
            "instrument_type": "Call option" if row["option_type"] == "call" else "Put option",
            "instrument_id": row["contract_symbol"], "underlying_symbol": inventory_symbol,
            "strike": row["strike"], "expiration": row["expiration"],
            "valid_quote_days": row["valid_quote_days"], "equity_price_days": None,
            "source": "ThetaData option history", "selection": "final sampled survivor",
        })
instrument_inventory = pd.DataFrame(inventory_rows, columns=[
    "instrument_type", "instrument_id", "underlying_symbol", "strike", "expiration",
    "valid_quote_days", "equity_price_days", "source", "selection"])
inventory_counts = instrument_inventory.groupby("instrument_type").size()
instrument_summary = pd.DataFrame([
    {"instrument_type": "Equity", "count": int(inventory_counts.get("Equity", 0)),
     "status": "price series available" if not equity_prices.is_empty() else "no prices in selected window"},
    {"instrument_type": "Call option", "count": int(inventory_counts.get("Call option", 0)), "status": "final sampled contracts"},
    {"instrument_type": "Put option", "count": int(inventory_counts.get("Put option", 0)), "status": "final sampled contracts"},
    {"instrument_type": "Corporate bond", "count": 0, "status": "not integrated into this workflow"},
    {"instrument_type": "Other traded instruments", "count": 0, "status": "not integrated into this workflow"},
])
print(f"{inventory_symbol} instrument inventory for {EDA_YEAR}: {len(instrument_inventory)} instruments.")
display(instrument_summary)
with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.max_colwidth", None):
    display(instrument_inventory)


AAPL instrument inventory for 2024: 11 instruments.


,instrument_type,count,status
0,Equity,1,price series available
1,Call option,5,final sampled contracts
2,Put option,5,final sampled contracts
3,Corporate bond,0,not integrated into this workflow
4,Other traded instruments,0,not integrated into this workflow


,instrument_type,instrument_id,underlying_symbol,strike,expiration,valid_quote_days,equity_price_days,source,selection
0,Equity,AAPL,AAPL,NaN,NaT,NaN,252.0,FMP equity history,equity price series
1,Call option,AAPL240719C00220000,AAPL,220.0,2024-07-19,138.0,NaN,ThetaData option history,final sampled survivor
2,Call option,AAPL240920C00155000,AAPL,155.0,2024-09-20,182.0,NaN,ThetaData option history,final sampled survivor
3,Call option,AAPL241220C00120000,AAPL,120.0,2024-12-20,246.0,NaN,ThetaData option history,final sampled survivor
4,Call option,AAPL241220C00135000,AAPL,135.0,2024-12-20,246.0,NaN,ThetaData option history,final sampled survivor
5,Call option,AAPL241220C00210000,AAPL,210.0,2024-12-20,246.0,NaN,ThetaData option history,final sampled survivor
6,Put option,AAPL240315P00185000,AAPL,185.0,2024-03-15,52.0,NaN,ThetaData option history,final sampled survivor
7,Put option,AAPL240315P00195000,AAPL,195.0,2024-03-15,52.0,NaN,ThetaData option history,final sampled survivor
8,Put option,AAPL240315P00200000,AAPL,200.0,2024-03-15,52.0,NaN,ThetaData option history,final sampled survivor
9,Put option,AAPL240419P00200000,AAPL,200.0,2024-04-19,76.0,NaN,ThetaData option history,final sampled survivor


## Step 2 — Understand what is loaded, and when

1. Discover equity profiles and price histories for the requested threshold, and option-date coverage for every eligible symbol.
2. Load issuer financial families, sparse events, macro series, and peer context from the warehouse. Peer context needs a cross-universe source pass on first use, so startup grows with universe size.
3. Assemble annual tensors only as training requests documents. The source cache is bounded and one CPU batch is prefetched while the GPU works.

There is no corpus-building command, prior-run roster, feature export, or fitted normalization pass. A packaged **field-name schema** defines the architecture; it contains no training examples. Numeric values use the fixed transform `sign(x) * log1p(abs(x)) / 10`, with missingness preserved.

“On demand” does not mean zero preparation: discovery, source reads, peer context, and the first batch precede the first optimizer update. Recorded first updates were 27.46 seconds for the completed $1T run and 123.11 seconds for the later $100B attempt. These are measurements of those runs, not promises for $10B.

## Step 3 — Build the synthetic option price series

On the **actual first NYSE trading session of each year**, select up to five positive-DTE expirations per right across the available range. With fewer than five expirations, use all available ones. Each selected call or put expiration becomes a basket of **all its strikes**, initially equal weighted.

The basket's membership is frozen for that year. Its time series follows those selected contracts; later contracts are not added, and there is no roll. These are historical option-price series, not theoretical prices inferred from equities. Basket terms and the underlying issuer/equity context accompany the option's own price stream.

- Missing first-session chains are recorded and that year is skipped, without selecting a later chain.
- A daily basket quote requires every constituent; missing members are not renormalized away.
- Forward splits adjust strikes and contract counts to preserve economic exposure.
- Expiry uses split-consistent intrinsic value from the underlying spot in the option chain. Missing required settlement values fail backtesting.
- A basket with no complete quote fails the run. Reverse-split deliverables require an explicit mapping.

Coverage can differ by year even when an underlying has stored options history. The report below exposes these differences.

## Step 4 — From fields to subtokens, tokens, and documents

| Level | Meaning in this run |
|---|---|
| Raw feature | An observed price, financial field, basket term, or sparse event value, with its date and missingness |
| Family subtoken | A learned vector encoding one feature family's observed fields; the family is **not averaged into one scalar** |
| Token | A learned representation combining family information and available time-aligned context |
| Annual document | One instrument's native observations within a calendar year, plus a memory-prefix position |
| Annual memory | Detached learned state passed from an earlier document to a later year of the **same instrument** |

For an equity, 2024 covers January 1–December 31, and 2025 starts January 1 with the state left by 2024. January 1 need not be a trading day. Daily, quarterly, annual, and sparse streams keep their native observation dates inside that annual document; quarterly financial observations do not create separate quarterly training documents.

Every newly formed annual option basket has its own identity, so its state does **not** carry into next year's newly selected basket. Equities retain their identity across years. Instruments are interleaved for batching while preserving each instrument's chronological order. Gradients do not backpropagate across years through detached memory.

An epoch visits all generated documents. Thus equity count is not sample count: historical years contribute equity documents, and each underlying/year can contribute up to ten option documents. Sparse option history can yield short documents; the final coverage gate requires temporal option documents for every eligible option underlying.

## Step 5 — What the optimizer learns

The run uses a 64-dimensional, two-layer multi-rate Transformer with four attention heads, FP32 CUDA, AdamW, batch size 16, and seed 0. Document-level tasks and MRL are disabled in this workflow.

Supervised Oracle/HITS targets come from each instrument's own price path; equity documents can also use warehouse event targets. Classification supervision is restricted to actual labeled events: missing labels are masked, not converted into “no event.” Option documents contribute their own targets.

Both next-token prediction and masked reconstruction train on observed feature sequences, with reconstruction weight 0.1. Each batch performs forward evaluation, weighted loss calculation, backpropagation, gradient clipping, and an optimizer update. Finite-loss/gradient checks stop invalid training. Checkpoints record model, optimizer, annual memory, normalization, and objective-observation counts.

Current warehouse-stream checkpoints are saved for audit; this loader does not implement resume or standalone inference from them. A fresh launch starts new weights. Backtests run inside the same process after each completed epoch.

In [5]:
def training_command(universe, output):
    return [sys.executable, str(ROOT / "scripts/train_multirate_mtl.py"),
        "--min-market-cap", str(MARKET_CAPS[universe]), "--output-dir", str(output),
        "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
        "--d-model", "64", "--num-heads", "4", "--layers", "2",
        "--mrl-dimensions", "", "--disable-document-tasks", "--device", "cuda",
        "--warehouse-start-date", "1900-01-01", "--train-end-date", TRAIN_END,
        "--prediction-start-date", PREDICTION_START, "--prediction-end-date", PREDICTION_END,
        "--checkpoint-every-batches", "10", "--progress-every-batches", "1",
        "--self-supervision", "both", "--reconstruction-weight", "0.1",
        "--sequence-mode", "annual_memory", "--skip-embeddings", "--skip-t-sne"]

# Preview is read-only. It does not query data, create a corpus, or launch training.
for universe in UNIVERSES:
    print(universe, shlex.join(training_command(universe, EXPERIMENT / universe / "NEW_RUN")), "\n")

1T /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python /home/jlee153232/PycharmProjects/quant-orchestrator/scripts/train_multirate_mtl.py --min-market-cap 1000000000000 --output-dir /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/NEW_RUN --epochs 1 --batch-size 16 --d-model 64 --num-heads 4 --layers 2 --mrl-dimensions '' --disable-document-tasks --device cuda --warehouse-start-date 1900-01-01 --train-end-date 2024-01-01 --prediction-start-date 2024-01-01 --prediction-end-date 2026-09-09 --checkpoint-every-batches 10 --progress-every-batches 1 --self-supervision both --reconstruction-weight 0.1 --sequence-mode annual_memory --skip-embeddings --skip-t-sne 

100B /home/jlee153232/miniconda3/envs/quant-orchestrator/bin/python /home/jlee153232/PycharmProjects/quant-orchestrator/scripts/train_multirate_mtl.py --min-market-cap 100000000000 --output-dir /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/NEW_RUN -

## Step 6 — Run training, or select existing outputs

In `train` mode this cell runs the existing trainer as a subprocess using the notebook kernel's Python, streams its logs, and waits for training **and** backtests. Each universe runs sequentially; a failed run stops the sequence with the traceback. Interrupting this cell terminates only the process it launched. Avoid launching another copy on the same GPU while this cell is active.

In `review` mode this cell selects existing warehouse-stream outputs without loading any market data. It prefers the latest run, even if failed, so failures are visible rather than hidden by an older success. Set `REVIEW_RUNS` to inspect a particular run. Re-running the training cell creates fresh runs; it never overwrites or resumes a prior run.

In [6]:
def read_json(path):
    return json.loads(path.read_text()) if path.is_file() else {}

RUNS = {}
if MODE == "train":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA-enabled kernel before training.")
    if importlib.util.find_spec("quant_warehouse") is None:
        raise RuntimeError("Install the repository's quant-warehouse dependency in this kernel.")
    env = os.environ.copy()
    env.update(POLARS_MAX_THREADS="8", OMP_NUM_THREADS="4", PYTHONUNBUFFERED="1")
    env["PYTHONPATH"] = str(ROOT) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    for universe in UNIVERSES:
        name = "warehouse_stream_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
        output = EXPERIMENT / universe / name
        output.parent.mkdir(parents=True, exist_ok=True)  # Trainer creates the run directory.
        RUNS[universe] = output
        command = training_command(universe, output)
        print(f"Starting {universe}: {output}", flush=True)
        with Path(str(output) + ".log").open("w") as log:
            process = subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            try:
                for line in process.stdout:
                    log.write(line); log.flush()
                    print(line, end="", flush=True)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill(); process.wait()
                raise
        if returncode:
            failure = output / "failure.txt"
            raise RuntimeError(f"Run failed: {output}\n" +
                               (failure.read_text() if failure.exists() else f"See {output}.log"))
        if read_json(output / "status.json").get("stage") != "complete":
            raise RuntimeError(f"Process exited without a complete status: {output}")
else:
    for universe in UNIVERSES:
        if universe in REVIEW_RUNS:
            candidate = Path(REVIEW_RUNS[universe]).expanduser()
            output = candidate if candidate.is_absolute() else ROOT / candidate
            if not (output / "configuration.json").is_file():
                raise ValueError(f"Not a run directory: {output}")
        else:
            candidates = sorted((EXPERIMENT / universe).glob("warehouse_stream_*/configuration.json"))
            candidates = [p.parent for p in candidates
                          if read_json(p).get("dataset_mode") == "warehouse_on_demand"]
            output = candidates[-1] if candidates else None
        if output is not None:
            RUNS[universe] = output
        else:
            print(f"{universe}: no warehouse-stream run found; choose train mode to create one.")

display(pd.DataFrame([{"universe": u, "run": str(p),
                       "stage": read_json(p / "status.json").get("stage", "status unavailable")}
                      for u, p in RUNS.items()]))

10B: no warehouse-stream run found; choose train mode to create one.


,universe,run,stage
0,1T,/home/jlee153232/PycharmProjects/quant-orchest...,complete
1,100B,/home/jlee153232/PycharmProjects/quant-orchest...,failed


## Step 7 — Confirm actual equity and option coverage

Read the run's **own** configuration, universe, and coverage outputs. `trained` counts reflect the last coverage write and can lag the live status by a few batches. Counts accumulate across epochs. An underlying listed in the selected universe is not proof that its option documents trained; check actual observation counts and the final run status.

The completed $1T run included 13 equities and 11 option underlyings. Berkshire's two share classes had no stored option series. The later $100B attempt reached batch 101 before stopping on `CVS/2021` because the four-day put basket had no complete quote. No success or $10B timing is inferred from those attempts.

In [7]:
coverage_rows, option_rows, cohort_rows = [], [], []
for universe, output in RUNS.items():
    config = read_json(output / "configuration.json")
    roster = read_json(output / "universe.json")
    coverage = read_json(output / "option_coverage.json")
    status = read_json(output / "status.json")
    startup = read_json(output / "startup_timing.json")
    if config.get("min_market_cap") != MARKET_CAPS[universe]:
        raise ValueError(f"Universe/path mismatch: {output}")
    trained = coverage.get("trained", {})
    coverage_rows.append({"universe": universe, "stage": status.get("stage"),
        "equities": len(roster.get("equity_symbols", [])),
        "expected_option_underlyings": len(roster.get("option_symbols", [])),
        "observed_option_underlyings": len(trained),
        "option_documents": sum(r.get("documents", 0) for r in trained.values()),
        "option_price_observations": sum(r.get("price_observations", 0) for r in trained.values()),
        "first_update_seconds": startup.get("first_optimizer_update_seconds"),
        "corpus_built": startup.get("corpus_built")})
    for symbol, row in trained.items():
        option_rows.append({"universe": universe, "symbol": symbol, **row})
    for symbol, detail in coverage.get("coverage", {}).items():
        for row in detail.get("cohorts", []):
            cohort_rows.append({"universe": universe, "symbol": symbol, **row})
    missing = sorted(set(roster.get("equity_symbols", [])) - set(roster.get("option_symbols", [])))
    print(f"{universe}: no eligible stored pre-cutoff options: {missing}")
    if (output / "failure.txt").exists():
        print(f"{universe} failure: " + (output / "failure.txt").read_text().strip().splitlines()[-1])
coverage_summary = pd.DataFrame(coverage_rows)
option_coverage = pd.DataFrame(option_rows)
cohorts = pd.DataFrame(cohort_rows)
display(coverage_summary)
if not option_coverage.empty:
    display(option_coverage.groupby("universe")[["documents", "price_observations", "temporal_documents"]].sum())
if not cohorts.empty:
    display(cohorts.groupby(["universe", "status"]).size().rename("underlying_years").to_frame())
# Inspect option_coverage or cohorts directly for every symbol/year; no truncated source data is used in training.

1T: no eligible stored pre-cutoff options: ['BRK-A', 'BRK-B']
100B: no eligible stored pre-cutoff options: ['ABALX', 'BALFX', 'BNY', 'BRK-A', 'BRK-B', 'FNPFX', 'RICAX', 'RICBX', 'SOJE', 'TBB', 'TBC', 'VZA']
100B failure: ValueError: CVS/2021: frozen baskets without any complete quote: ['OPT_CVS_2021_PUT_DTE_4']


,universe,stage,equities,expected_option_underlyings,observed_option_underlyings,option_documents,option_price_observations,first_update_seconds,corpus_built
0,1T,complete,13,11,11,790,51218,27.460330,False
1,100B,failed,127,115,21,895,42408,123.105659,False


,documents,price_observations,temporal_documents
universe,,,
100B,895,42408,762
1T,790,51218,744


underlying_years
universe status                                       
100B     loaded                                     95
         missing_first_session_chain                53
1T       loaded                                    112
         missing_first_session_chain                36

## Step 8 — Score the held-out years without warmup

After each epoch, inference starts with empty annual memory at the requested start date. It does not replay pre-cutoff history. Memory can carry between requested years of the same equity. Only dates with observed instrument prices become trading scores; issuer/macro context alone cannot create a tradeable day.

The trainer checks duplicate predictions, omitted priced dates, and nonfinite scores. Predictions and prices are then saved as **backtest outputs**, never reused as a fresh training corpus.

Financial features retain warehouse-recorded observation dates. No additional reporting lag or historical data-vintage reconstruction is applied. The selected universe also comes from stored profiles. These are research timing/universe conventions, not a claim of a fully reconstructed point-in-time investable universe.

## Step 9 — Understand the four backtest books

| Book | Decisions and exposure |
|---|---|
| Equity long | Existing HITS strategy, long equity positions |
| Equity short | Existing HITS strategy, short equity positions |
| Long calls | Bullish **equity** signals buy the year's frozen call baskets |
| Long puts | Bearish **equity** signals buy the year's frozen put baskets |

Each year/book starts from its own $100,000. Returns are not additive across books. Options use equity signals as the decision maker even though the model also scores option instruments. Option orders execute no earlier than the next equity session and require complete basket quotes. The capacity planner holds positions awaiting an executable exit; expiry forces settlement without rolling.

The shared fixed-weight return engine applies 5.5 bps per unit of turnover; options also incur modeled bid/ask spread costs. Missing option marks carry forward **for valuation only**, and reports count stale position-days and open positions at period end. This is not an exact broker cash/margin ledger.

The table below uses `capital_return` (final equity / starting cash − 1), including the initial day's capital effect. It does not substitute the equity report's differently based `total_return`. A missing report is shown as missing, never zero.

In [8]:
report_rows, timing_rows = [], []
for universe, output in RUNS.items():
    files = sorted((output / "epoch_validation").glob("epoch_*/results.json"))
    if not files:
        print(f"{universe}: no completed epoch backtests in {output.name}")
    for path in files:
        epoch = int(path.parent.name.split("_")[-1])
        reports = json.loads(path.read_text())
        for report in reports:
            row = {"universe": universe, "epoch": epoch, **report}
            row["book"] = ("equity_" + report["side"] if report["asset_class"] == "equity" else report["side"])
            report_rows.append(row)
        timing_rows.append({"universe": universe, "epoch": epoch, **read_json(path.parent / "timing.json")})
results = pd.DataFrame(report_rows)
if not results.empty:
    returns = results.pivot(index=["universe", "epoch", "year"], columns="book", values="capital_return")
    returns = returns.reindex(columns=["equity_long", "equity_short", "long_calls", "long_puts"])
    display(returns.style.format("{:+.2%}", na_rep="missing"))
    detail_columns = ["universe", "epoch", "year", "book", "sharpe", "max_drawdown", "entries",
                      "stale_valuation_position_days", "open_positions_at_period_end"]
    display(results.reindex(columns=detail_columns))
if timing_rows:
    display(pd.DataFrame(timing_rows))

100B: no completed epoch backtests in warehouse_stream_20260913T032457Z


,universe,epoch,year,book,sharpe,max_drawdown,entries,stale_valuation_position_days,open_positions_at_period_end
0,1T,1,2024,equity_long,1.343739,-0.114361,169,NaN,NaN
1,1T,1,2024,equity_short,-2.477028,-0.200859,163,NaN,NaN
2,1T,1,2024,long_calls,0.485548,-0.345860,302,196.0,9.0
3,1T,1,2024,long_puts,-2.724612,-0.828264,305,577.0,6.0
4,1T,1,2025,equity_long,1.126882,-0.129504,207,NaN,NaN
5,1T,1,2025,equity_short,-1.415826,-0.201314,195,NaN,NaN
6,1T,1,2025,long_calls,0.179244,-0.526800,359,32.0,5.0
7,1T,1,2025,long_puts,-1.659229,-0.552280,336,59.0,7.0
8,1T,1,2026,equity_long,1.353587,-0.082731,137,NaN,NaN
9,1T,1,2026,equity_short,-0.960276,-0.117429,126,NaN,NaN


,universe,epoch,inference_seconds,backtest_seconds,predictions,inference_initialization
0,1T,1,144.429703,1.742271,41139,empty_memory_no_warmup


## Step 10 — Locate checkpoints, failures, and detailed artifacts

A completed run must have a complete status, a final epoch checkpoint, both trained asset classes, and all four books for each requested year. The checks below apply to completed runs only. A failed run is retained for diagnosis and is not silently replaced with an older result.

`cohort_members/` contains frozen membership audit outputs. `epoch_validation/epoch_XXXX/` contains predictions, price snapshots, timings, and year/asset reports. The `.log` alongside the run records batch-level progress. Keep the run path to reproduce or inspect a specific result.

To compare another threshold, change Step 1 and rerun from there. The same feature, model, and backtest code applies at every threshold. Larger universes may expose warehouse gaps, as the $100B attempt did; choosing $10B is supported but is not evidence that a full $10B options run has already passed.

In [9]:
for universe, output in RUNS.items():
    status = read_json(output / "status.json")
    print(f"\n{universe}: {output}")
    print("Log:", Path(str(output) + ".log"))
    print("Coverage:", output / "option_coverage.json")
    print("Failure details:", output / "failure.txt" if (output / "failure.txt").exists() else "none recorded")
    if status.get("stage") != "complete":
        print("Not marked complete; no completed-run assertion is made.")
        continue
    config = read_json(output / "configuration.json")
    final_epoch = int(config["epochs"])
    assert (output / f"epoch_{final_epoch:04d}.pt").is_file()
    assert status["documents"].get("equity", 0) > 0 and status["documents"].get("option", 0) > 0
    roster = read_json(output / "universe.json")
    trained = read_json(output / "option_coverage.json")["trained"]
    assert set(roster["option_symbols"]) <= set(trained)
    assert all(trained[s]["temporal_documents"] > 0 for s in roster["option_symbols"])
    years = range(int(config["prediction_start_date"][:4]), int(config["prediction_end_date"][:4]) + 1)
    expected = {(year, asset, side) for year in years for asset, sides in
                [("equity", ("long", "short")), ("option", ("long_calls", "long_puts"))] for side in sides}
    for epoch in range(1, final_epoch + 1):
        report_path = output / "epoch_validation" / f"epoch_{epoch:04d}" / "results.json"
        reports = json.loads(report_path.read_text())
        assert len(reports) == len(expected)
        assert {(r["year"], r["asset_class"], r["side"]) for r in reports} == expected
    print("Completed-run artifact and coverage checks passed.")


1T: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z
Log: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z.log
Coverage: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/1T/warehouse_stream_20260912T201813Z/option_coverage.json
Failure details: none recorded
Completed-run artifact and coverage checks passed.

100B: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z
Log: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z.log
Coverage: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse_stream_20260913T032457Z/option_coverage.json
Failure details: /home/jlee153232/PycharmProjects/quant-orchestrator/artifacts/multirate_recovery/100B/warehouse

### Implementation references

- [Warehouse documents and coverage](../quant_orchestrator/research_tools/warehouse_multirate.py)
- [Shared training objectives](../quant_orchestrator/research_tools/multirate_training_step.py)
- [Streaming training and epoch evaluation](../quant_orchestrator/research_tools/warehouse_multirate_training.py)
- [Frozen baskets and split adjustments](../quant_orchestrator/research_tools/frozen_option_adjustments.py)
- [Option backtest](../quant_orchestrator/platforms/backtesting_frameworks/frozen_option_backtest.py)
- [Workflow documentation](../docs/multirate-warehouse-streaming.md)

The saved outputs are an executed artifact review, not a new training run performed by this notebook.